# Assignment 2 Question 3

## d) Simulate the data

In [1]:
import numpy as np

np.random.seed(12345)

n, p = 200, 8  
X = np.random.randn(n, p)
true_beta =[2,-1.5,0,3,0,-2.5,1,5]

y = X @ true_beta + 0.5 * np.random.randn(n)  # Adding some noise

In [2]:
lambda_=0.2
tol=1e-6

The the maximum step size to ensure convergence is equal to $\frac{1}{\mu}=\frac{1}{\lambda\text{max}(\frac{1}{n}\mathbf{X'X}+\lambda)}$.

In [3]:
eigenvalues, eigenvectors = np.linalg.eig((X.T @ X)/n)
mu=max(eigenvalues)+lambda_
t=1/mu
print(f"The largest step size: {t:.4f}")

The largest step size: 0.6520


The upper bound on the suboptimality gap, $\mathcal{R}(\beta^{(i)})-\mathcal{R}(\beta^*)$, under the stopping condition, $||(\nabla_\beta\mathcal{R})(\beta)||_2\leq\epsilon$ is equal to $\frac{\epsilon^2}{2\tau}=\frac{(10^{-6})^2}{2(\lambda\text{min}(\frac{1}{n}\mathbf{X'X})+\lambda)}$.

In [4]:
tau=min(eigenvalues)+lambda_
print(f"\u03c4: {tau:.4f}")
print("The upper bound on the suboptimality gap: ",(tol**2)/(2*tau)) 

τ: 0.9223
The upper bound on the suboptimality gap:  5.421044018712684e-13


## f) Gradient decent

In [5]:
import pandas as pd
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0, ddof=0)
y_mean = np.mean(y)
    
X_std[X_std == 0] = 1  # To avoid division by zero for constant columns
X_scaled = (X - X_mean) / X_std
y_centered = y - y_mean

beta_star=np.linalg.inv(X_scaled.T@X_scaled+n*lambda_*np.identity(p)) @ (X_scaled.T@y_centered)

df=pd.DataFrame(true_beta,columns=["True Beta"])
df["Beta_star"]=beta_star

In [ ]:
def gradient_decent(y,X,lambda_,beta_0,t,max_iter,tol):
    n,p=X.shape
    Q=(X.T@X)/n-(X.T@y)/n
    beta_prev=beta_0
    gradient=Q+lambda_*beta_prev
    for k in range(max_iter):
        beta_hat=beta_prev-t*gradient
        gradient=Q+lambda_*beta_hat
        norm=np.linalg.norm(gradient)
        if (norm<=tol):
            return (beta_hat,k,norm)
        beta_prev=beta_hat

    return (beta_hat,k,norm)


In [15]:

coef,iterations,norm=gradient_decent(y_centered,X_scaled,lambda_,np.zeros(p),t,1000,tol)
#df["beta_k"]=coef

print("Gradient descent results:")
print("Number of iterations (k): ",iterations)
print("Final gradient norm: ",norm)
print(df)



[[-1.12725498  0.75220532 -0.74209877 -2.91300244  0.41359205  2.99568187
  -0.47984871 -4.94667676]
 [-2.05608488  1.68103522 -0.93470231 -2.94122136  0.39546334  3.05730098
  -0.3622196  -4.83089762]
 [-2.0000584   0.61562827  0.13070465 -3.01011695  0.34320788  3.00332692
  -0.42384424 -4.82629764]
 [-2.03580192  0.74426937 -0.8749568  -2.00445551  0.38256435  3.00900253
  -0.63120607 -4.88133754]
 [-2.15031748  0.639844   -0.96274203 -3.05854571  1.43665455  3.08055728
  -0.53169466 -4.91041074]
 [-2.2367663   0.63314301 -0.97116162 -3.10064617  0.41201865  4.10519319
  -0.55433382 -4.92245375]
 [-2.0745205   0.85139881 -0.7605564  -3.10307839  0.43754308  3.08344256
   0.46741681 -4.86209937]
 [-2.21265589  0.71141345 -0.83431714 -3.0245172   0.38751966  3.04401529
  -0.53340672 -3.86127585]]
Gradient descent results:
Number of iterations (k):  120
Final gradient norm:  8.881974041510554e-07
   True Beta  Beta_star
0        2.0   1.715485
1       -1.5  -1.073638
2        0.0   0.1